In [1]:
from simulation import *

random.seed(0)

sim = Simulation("../input-S1-14.txt", 100, 100, 1)
print(sim.patients)
sim.generatePatients()
print(sim.patients)

[]
[<patient.Patient object at 0x00000214D41576D0>, <patient.Patient object at 0x00000214D42DA090>, <patient.Patient object at 0x00000214D42D9FD0>, <patient.Patient object at 0x00000214D42DA050>, <patient.Patient object at 0x00000214D42D9F90>, <patient.Patient object at 0x00000214D42DA150>, <patient.Patient object at 0x00000214D42DA190>, <patient.Patient object at 0x00000214D42DA1D0>, <patient.Patient object at 0x00000214D42DA210>, <patient.Patient object at 0x00000214D42DA110>, <patient.Patient object at 0x00000214D42DA250>, <patient.Patient object at 0x00000214D42DA290>, <patient.Patient object at 0x00000214D42DA2D0>, <patient.Patient object at 0x00000214D42DA310>, <patient.Patient object at 0x00000214D42DA350>, <patient.Patient object at 0x00000214D42DA390>, <patient.Patient object at 0x00000214D42DA3D0>, <patient.Patient object at 0x00000214D42DA410>, <patient.Patient object at 0x00000214D42DA450>, <patient.Patient object at 0x00000214D42DA490>, <patient.Patient object at 0x0000021

In [2]:
import sys, random
import pandas as pd
sys.path.insert(0, '.')
from simulation import Simulation

PATIENT_TYPES = {1: 'elective', 2: 'urgent'}
SCAN_TYPES    = {0: 'brain', 1: 'lumbar', 2: 'cervical', 3: 'abdomen', 4: 'other'}
DAYS          = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat'}

def patient_repr(p) -> str:
    ptype = PATIENT_TYPES.get(p.patientType, '?')
    stype = SCAN_TYPES.get(p.scanType, '?') if p.patientType == 2 else '-'
    call  = f"W{p.callWeek} {DAYS.get(p.callDay,'?')} {p.callTime:.2f}h"
    appt  = 'unscheduled' if p.scanWeek == -1 else \
            f"W{p.scanWeek} {DAYS.get(p.scanDay,'?')} slot={p.slotNr} @{p.appTime:.2f}h"
    scan  = f"scanTime={p.scanTime:.2f}h" if p.scanTime != -1 else 'not yet scanned'
    tard  = f"tard={p.tardiness*60:+.1f}min"
    ns    = ' NO-SHOW' if p.isNoShow else ''
    return f"[{p.nr:>4}] {ptype:<8} {stype:<8}  arrived: {call:<22}  appt: {appt:<30}  {scan:<22}  {tard}{ns}"

def print_patients(patients, limit=None):
    shown = patients[:limit] if limit else patients
    for p in shown:
        print(patient_repr(p))
    if limit and len(patients) > limit:
        print(f'  ... ({len(patients) - limit} more)')

def patients_to_df(patients) -> pd.DataFrame:
    rows = []
    for p in patients:
        rows.append({
            'nr':            p.nr,
            'type':          PATIENT_TYPES.get(p.patientType, p.patientType),
            'scan_type':     SCAN_TYPES.get(p.scanType, p.scanType),
            'call_week':     p.callWeek,
            'call_day':      DAYS.get(p.callDay, p.callDay),
            'call_time':     p.callTime,
            'scan_week':     p.scanWeek,
            'scan_day':      DAYS.get(p.scanDay, p.scanDay) if p.scanDay != -1 else None,
            'slot_nr':       p.slotNr,
            'app_time':      p.appTime,
            'tardiness_min': p.tardiness * 60,
            'no_show':       p.isNoShow,
            'scan_time':     p.scanTime if p.scanTime != -1 else None,
            'duration_min':  p.duration * 60,
        })
    return pd.DataFrame(rows)


In [3]:
print_patients(sim.patients, limit=10)

[   0] elective -         arrived: W0 Mon 8.05h            appt: unscheduled                     not yet scanned         tard=-0.6min
[   1] elective -         arrived: W0 Mon 8.07h            appt: unscheduled                     not yet scanned         tard=+2.5min
[   2] elective -         arrived: W0 Mon 8.28h            appt: unscheduled                     not yet scanned         tard=+0.8min
[   3] elective -         arrived: W0 Mon 9.00h            appt: unscheduled                     not yet scanned         tard=+0.7min
[   4] elective -         arrived: W0 Mon 9.26h            appt: unscheduled                     not yet scanned         tard=-3.5min
[   5] elective -         arrived: W0 Mon 9.46h            appt: unscheduled                     not yet scanned         tard=-1.6min
[   6] elective -         arrived: W0 Mon 9.58h            appt: unscheduled                     not yet scanned         tard=+3.9min
[   7] elective -         arrived: W0 Mon 9.69h            app

In [4]:
SLOT_LABELS = {0: '.', 1: 'E', 2: 'U', 3: 'T'}
DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

def print_week_schedule_compact(sim):
    print(f"{'slot':>4}  {'time':>5}  " + " ".join(f"{d:>3}" for d in DAYS))
    for s in range(sim.S):
        slot0 = sim.weekSchedule[0][s]
        if not hasattr(slot0, 'startTime'):
            print("weekSchedule not populated yet — call sim.setWeekSchedule() first")
            return
        time_str = f"{slot0.startTime:.2f}h"
        labels = " ".join(f"{SLOT_LABELS.get(sim.weekSchedule[d][s].slotType, '?'):>3}" for d in range(sim.D))
        print(f"{s:>4}  {time_str:>5}  {labels}")


In [5]:
print_week_schedule_compact(sim)
sim.setWeekSchedule()
print_week_schedule_compact(sim)

slot   time  Mon Tue Wed Thu Fri Sat
weekSchedule not populated yet — call sim.setWeekSchedule() first
slot   time  Mon Tue Wed Thu Fri Sat
   0  8.00h    E   E   E   E   E   E
   1  8.25h    E   E   E   E   E   E
   2  8.50h    E   E   E   E   E   E
   3  8.75h    E   E   E   E   E   E
   4  9.00h    E   E   E   E   E   E
   5  9.25h    E   E   E   E   E   E
   6  9.50h    E   E   E   E   E   E
   7  9.75h    E   E   E   E   E   E
   8  10.00h    E   E   E   E   E   E
   9  10.25h    E   E   E   E   E   E
  10  10.50h    E   E   E   E   E   E
  11  10.75h    E   E   E   E   E   E
  12  11.00h    E   E   E   E   E   E
  13  11.25h    E   E   E   E   E   E
  14  11.50h    U   U   U   E   U   E
  15  11.75h    U   U   U   U   U   U
  16  13.00h    E   E   E   .   E   .
  17  13.25h    E   E   E   .   E   .
  18  13.50h    E   E   E   .   E   .
  19  13.75h    E   E   E   .   E   .
  20  14.00h    E   E   E   .   E   .
  21  14.25h    E   E   E   .   E   .
  22  14.50h    E   E   E   .   

In [6]:
# Step 3: schedule patients
sim.schedulePatients()

print("After scheduling:")
print_patients(sim.patients, limit=30)


IndexError: list index out of range

In [ ]:
n_view = 10

random.seed(0)

sim.setWeekSchedule()

sim.resetSystem()

sim.generatePatients()
print_patients(sim.patients, limit=n_view)

sim.schedulePatients()

print_patients(sim.patients, limit=n_view)

[   0] elective -         arrived: W0 Mon 8.05h            appt: unscheduled                     not yet scanned         tard=-0.6min
[   1] elective -         arrived: W0 Mon 8.07h            appt: unscheduled                     not yet scanned         tard=+2.5min
[   2] elective -         arrived: W0 Mon 8.28h            appt: unscheduled                     not yet scanned         tard=+0.8min
[   3] elective -         arrived: W0 Mon 9.00h            appt: unscheduled                     not yet scanned         tard=+0.7min
[   4] elective -         arrived: W0 Mon 9.26h            appt: unscheduled                     not yet scanned         tard=-3.5min
[   5] elective -         arrived: W0 Mon 9.46h            appt: unscheduled                     not yet scanned         tard=-1.6min
[   6] elective -         arrived: W0 Mon 9.58h            appt: unscheduled                     not yet scanned         tard=+3.9min
[   7] elective -         arrived: W0 Mon 9.69h            app

In [11]:
PATIENT_TYPES = {1: 'elective', 2: 'urgent'}
SCAN_TYPES    = {0: 'brain', 1: 'lumbar', 2: 'cervical', 3: 'abdomen', 4: 'other'}
DAYS          = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat'}  # dict!

def patient_repr(p) -> str:
    ptype = PATIENT_TYPES.get(p.patientType, '?')
    stype = SCAN_TYPES.get(p.scanType, '?') if p.patientType == 2 else '-'
    call  = f"W{p.callWeek} {DAYS.get(p.callDay,'?')} {p.callTime:.2f}h"
    appt  = 'unscheduled' if p.scanWeek == -1 else \
            f"W{p.scanWeek} {DAYS.get(p.scanDay,'?')} slot={p.slotNr} @{p.appTime:.2f}h"
    scan  = f"scanTime={p.scanTime:.2f}h" if p.scanTime != -1 else 'not yet scanned'
    tard  = f"tard={p.tardiness*60:+.1f}min"
    ns    = ' NO-SHOW' if p.isNoShow else ''
    return f"[{p.nr:>4}] {ptype:<8} {stype:<8}  arrived: {call:<22}  appt: {appt:<30}  {scan:<22}  {tard}{ns}"

def print_patients(patients, limit=None):
    shown = patients[:limit] if limit else patients
    for p in shown:
        print(patient_repr(p))
    if limit and len(patients) > limit:
        print(f'  ... ({len(patients) - limit} more)')


In [10]:
sim.patients

 ...]